# Laboratori 05: Diferencimi dhe integrimi numerik

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si zgjedhim hapin që balancon gabimin e këputjes me gabimin e rrumbullakimit?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- nxjerrë formulat përpara dhe qendrore nga zhvillimi Taylor;
- matë rendin e gabimit dhe të identifikojë hapin optimal;
- nxjerrë rregullat e trapezit dhe Simpson-it;
- numërojë vlerësimet e funksionit dhe kujtesën;
- dallojë konvergjencën formale nga qëndrueshmëria numerike;
- zbatojë një kuadraturë të përbërë dhe një kontroll Richardson.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Diferencimi dhe integrimi reagojnë ndryshe ndaj zhurmës. Derivati thekson ndryshimet e vogla dhe mund ta amplifikojë zhurmën me faktor $1/h$; integrali i mesatarizon luhatjet. Prandaj “rrjetë më e imët” nuk ka të njëjtin kuptim në të dy problemet. Çdo zgjedhje e hapit duhet të shoqërohet nga një model gabimi.

Në kod do të ndani tri burime: gabimin e formulës së diferencës ose kuadraturës, gabimin e rrumbullakimit dhe gabimin në vetë të dhënat. Formula me rend më të lartë ul gabimin e këputjes për funksione të lëmuara, por mund të kërkojë më shumë vlera dhe nuk e heq problemin e zhurmës.


## 1. Derivati si kufi dhe si llogaritje

Përkufizimi

$$f'(x)=\lim_{h\to0}\frac{f(x+h)-f(x)}{h}$$

nuk thotë se në kompjuter duhet zgjedhur $h$ sa më i vogël. Për $h$ të madh dominon gabimi i këputjes; për $h$ shumë të vogël zbriten numra pothuajse të barabartë dhe gabimi i rrumbullakimit përforcohet nga pjesëtimi me $h$.

Kjo është situata ideale për të dalluar analizën simbolike nga eksperimenti numerik.


## 2. Formula përpara me derivim të plotë

Nga Taylor-i,

$$f(x+h)=f(x)+hf'(x)+\frac{h^2}{2}f''(x)+O(h^3).$$

Duke zgjidhur për $f'(x)$:

$$D_+f(x)=\frac{f(x+h)-f(x)}{h}
=f'(x)+\frac{h}{2}f''(x)+O(h^2).$$

Prandaj gabimi dominues është $O(h)$. Për formulën qendrore, zbritja e zhvillimeve në $x+h$ dhe $x-h$ anulon termat çift:

$$D_0f(x)=\frac{f(x+h)-f(x-h)}{2h}
=f'(x)+\frac{h^2}{6}f^{(3)}(x)+O(h^4),$$

pra gabimi është $O(h^2)$.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


In [ ]:
def d_perpara(f,x,h): return (f(x+h)-f(x))/h
def d_qendrore(f,x,h): return (f(x+h)-f(x-h))/(2*h)

f=np.sin; df=np.cos; x0=1.0
for h in [1e-1,1e-2,1e-3]:
    e1=abs(d_perpara(f,x0,h)-df(x0))
    e2=abs(d_qendrore(f,x0,h)-df(x0))
    print(f'h={h:.0e}: përpara {e1:.3e}, qendrore {e2:.3e}')


In [ ]:
h=np.logspace(-16,-1,250)
e_p=np.abs(np.array([d_perpara(f,x0,hi) for hi in h])-df(x0))
e_q=np.abs(np.array([d_qendrore(f,x0,hi) for hi in h])-df(x0))

fig,ax=plt.subplots(figsize=(7,4))
ax.loglog(h,e_p,label='përpara $O(h)$'); ax.loglog(h,e_q,label='qendrore $O(h^2)$')
ax.set(xlabel='hapi $h$',ylabel='gabimi absolut',title='Këputja kundrejt rrumbullakimit')
ax.grid(True,which='both',alpha=.3); ax.legend(); plt.show()
print('h optimal qendror ≈',h[np.argmin(e_q)])


## 3. Vlerësimi i hapit optimal

Për formulën qendrore modelojmë gabimin total si

$$E(h)\approx C_1h^2+C_2\frac{\varepsilon_{\rm mach}}{h}.$$

Minimizimi, $E'(h)=0$, jep

$$h_{\rm opt}\sim\left(\frac{C_2\varepsilon_{\rm mach}}{2C_1}\right)^{1/3}.$$

Rregulli i shkallëzimit është $h_{\rm opt}\propto\varepsilon_{\rm mach}^{1/3}$. Konstanta varet nga funksioni dhe shkalla e $x$; prandaj nuk ka një hap universal.


In [ ]:
eps=np.finfo(float).eps
h_parashikuar=eps**(1/3)*(1+abs(x0))
print('eps^(1/3)*(1+|x|) =',h_parashikuar)
print('h i matur          =',h[np.argmin(e_q)])


## 4. Integrimi: rregulli i trapezit

Në një nëninterval $[x_i,x_{i+1}]$ me gjerësi $h$, funksioni zëvendësohet nga vija e drejtë midis skajeve. Sipërfaqja e trapezit është

$$I_i\approx\frac{h}{2}[f(x_i)+f(x_{i+1})].$$

Për $n$ nënintervale të barabarta:

$$T_n=h\left[\frac{f(a)+f(b)}{2}+\sum_{i=1}^{n-1}f(a+ih)\right].$$

Gabimi global është $O(h^2)=O(n^{-2})$. Numri i vlerësimeve të funksionit është $n+1$.


In [ ]:
def trapezi(f,a,b,n):
    x=np.linspace(a,b,n+1); h=(b-a)/n
    return h*(0.5*f(x[0])+np.sum(f(x[1:-1]))+0.5*f(x[-1]))

for n in [4,8,16,32,64]:
    I=trapezi(np.sin,0,np.pi,n)
    print(f'n={n:3d}, I={I:.12f}, gabimi={abs(I-2):.3e}')


## 5. Simpson-i nga interpolimi kuadratik

Në dy nënintervale përdorim vlerat në $x_0$, $x_1=x_0+h$ dhe $x_2=x_0+2h$. Integrimi i polinomit kuadratik jep

$$\int_{x_0}^{x_2}f(x)\,dx\approx\frac{h}{3}[f(x_0)+4f(x_1)+f(x_2)].$$

Për $n$ çift:

$$S_n=\frac{h}{3}\left[f_0+f_n+4\sum_{i\,\text{tek}}f_i+2\sum_{i\,\text{çift},\,i\ne0,n}f_i\right].$$

Gabimi global është $O(h^4)$. Kushti që $n$ të jetë çift nuk është hollësi programimi; vjen nga ndarja në çifte nënintervalesh.


In [ ]:
def simpson(f,a,b,n):
    if n%2: raise ValueError('n duhet të jetë çift.')
    x=np.linspace(a,b,n+1); h=(b-a)/n; y=f(x)
    return h/3*(y[0]+y[-1]+4*np.sum(y[1:-1:2])+2*np.sum(y[2:-1:2]))

for n in [4,8,16,32,64]:
    I=simpson(np.sin,0,np.pi,n)
    print(f'n={n:3d}, I={I:.15f}, gabimi={abs(I-2):.3e}')


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 6.1 Kompleksiteti me dorë

Të dy rregullat e përbëra vlerësojnë funksionin në $n+1$ pika dhe kryejnë një shumë lineare:

$$T(n)=c_fn+c_an+c_0\in\Theta(n).$$

Nëse pikat ndërtohen si vargje, kujtesa është $\Theta(n)$; një zbatim rrjedhës mund të përdorë $\Theta(1)$ kujtesë. Simpson-i ka rend më të lartë, kështu që për një tolerancë të njëjtë zakonisht kërkon më pak pika; klasa $\Theta(n)$ vetëm nuk e tregon këtë përfitim.

Nga $E_T\sim C_Tn^{-2}$ del $n\sim\varepsilon^{-1/2}$, ndërsa nga $E_S\sim C_Sn^{-4}$ del $n\sim\varepsilon^{-1/4}$.


In [ ]:
ns=2**np.arange(3,13)
tp=[]; ts=[]
for n in ns:
    t0=perf_counter(); trapezi(np.sin,0,np.pi,int(n)); tp.append(perf_counter()-t0)
    t0=perf_counter(); simpson(np.sin,0,np.pi,int(n)); ts.append(perf_counter()-t0)
print('pjerrësia trapez=',np.polyfit(np.log(ns[-5:]),np.log(tp[-5:]),1)[0])
print('pjerrësia Simpson=',np.polyfit(np.log(ns[-5:]),np.log(ts[-5:]),1)[0])


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 7.1 Richardson dhe kontrolli i gabimit

Nëse $I(h)=I+C h^p+O(h^{p+1})$, atëherë

$$I(h/2)=I+C\frac{h^p}{2^p}+O(h^{p+1}).$$

Eliminimi i $C$ jep përafrimin e përmirësuar

$$I_R=I(h/2)+\frac{I(h/2)-I(h)}{2^p-1}.$$

Diferenca $|I(h/2)-I(h)|/(2^p-1)$ shërben si vlerësim gabimi vetëm kur kemi hyrë në regjimin asimptotik. Nëse rrumbullakimi dominon, përgjysmimi i hapit nuk ndjek më ligjin $h^p$.


In [ ]:
def richardson_trapez(f,a,b,n):
    I1=trapezi(f,a,b,n); I2=trapezi(f,a,b,2*n)
    IR=I2+(I2-I1)/(2**2-1)
    est=abs(I2-I1)/(2**2-1)
    return I1,I2,IR,est

I1,I2,IR,est=richardson_trapez(np.exp,0,1,16)
sak=np.e-1
print('T_n=',I1,'T_2n=',I2,'Richardson=',IR)
print('vlerësim=',est,'gabim real=',abs(IR-sak))


## 8. Shembulli i plotë: integrali Gaussian

Llogarisim

$$I=\int_0^2e^{-x^2}\,dx.$$

Nuk kemi nevojë për një formulë elementare; një referencë me rrjetë shumë të imët ose funksion special mund të përdoret për verifikim. Punishtja ndryshon $n$, rregullin dhe intervalin. Për çdo rast regjistroni numrin e vlerësimeve dhe gabimin e vlerësuar.


In [ ]:
f_gauss=lambda x:np.exp(-x*x)
referenca=0.8820813907624216
ns=2**np.arange(2,9)
eT=np.array([abs(trapezi(f_gauss,0,2,int(n))-referenca) for n in ns])
eS=np.array([abs(simpson(f_gauss,0,2,int(n))-referenca) for n in ns])
fig,ax=plt.subplots(figsize=(7,4))
ax.loglog(ns,eT,'o-',label='trapezi'); ax.loglog(ns,eS,'s-',label='Simpson')
ax.set(xlabel='n',ylabel='gabimi absolut'); ax.grid(True,which='both',alpha=.3); ax.legend(); plt.show()


In [ ]:
def punishte_hapi(x0=1.0):
    h=np.logspace(-16,-1,200)
    ep=np.abs(np.array([d_perpara(np.sin,x0,z) for z in h])-np.cos(x0))
    eq=np.abs(np.array([d_qendrore(np.sin,x0,z) for z in h])-np.cos(x0))
    return h,ep,eq,h[np.argmin(eq)]

for x0 in [0.1,1.0,10.0]:
    _,_,_,hop=punishte_hapi(x0)
    print(f'x0={x0}: h optimal i matur={hop:.3e}')


### Njësitë dhe kontrolli fizik

Nëse $f$ ka njësi metre dhe $x$ sekonda, $f'$ duhet të ketë njësi $\mathrm{m/s}$, ndërsa $\int f\,dx$ njësi $\mathrm{m\,s}$. Humbja e faktorit $h$ në kuadraturë ose përdorimi i $2h$ gabimisht në diferencën qendrore shpesh zbulohet nga kontrolli dimensional para çdo prove numerike.


In [ ]:
# Rend i vëzhguar për diferencat para se rrumbullakimi të dominojë.
hs=np.array([.2,.1,.05,.025,.0125])
ep=np.array([abs(d_perpara(np.sin,1,h)-np.cos(1)) for h in hs])
eq=np.array([abs(d_qendrore(np.sin,1,h)-np.cos(1)) for h in hs])
print('rendet përpara:',np.log2(ep[:-1]/ep[1:]))
print('rendet qendrore:',np.log2(eq[:-1]/eq[1:]))


### Diferencimi i të dhënave me zhurmë

Le të kemi $y_i=f(x_i)+\eta_i$ me devijim standard $\sigma$. Në diferencën qendrore termi i zhurmës pjesëtohet me $2h$, kështu që shkalla e tij rritet afërsisht si $\sigma/h$. Zvogëlimi i hapit ul këputjen, por rrit këtë komponent. Në eksperimente reale hapi optimal varet edhe nga cilësia e matjes, jo vetëm nga epsiloni i makinës.


In [ ]:
def gabim_derivati_me_zhurme(h,sigma=.001,M=2000):
    yp=np.sin(1+h)+rng.normal(0,sigma,M); ym=np.sin(1-h)+rng.normal(0,sigma,M)
    d=(yp-ym)/(2*h)
    return np.sqrt(np.mean((d-np.cos(1))**2))
for h in [1e-1,3e-2,1e-2,3e-3,1e-3]:
    print(h,gabim_derivati_me_zhurme(h))


### Kontrolli i rendit të kuadraturës

Për trapezin, raporti $E_n/E_{2n}$ duhet t'i afrohet $4$; për Simpson-in duhet t'i afrohet $16$. Kur gabimi arrin saktësinë e pikës lundruese, raportet bëhen të paqëndrueshme. Raporti i rendit duhet llogaritur në intervalin ku gabimi bie rregullisht, jo duke përfshirë verbërisht pikat e fundit.


In [ ]:
for method,name,pord in [(trapezi,'trapezi',2),(simpson,'Simpson',4)]:
    E=[]
    for n in [8,16,32,64]: E.append(abs(method(np.exp,0,1,n)-(np.e-1)))
    print(name,'raportet=',np.array(E[:-1])/np.array(E[1:]),'pritja=',2**pord)


### Pyetje për vetëkontroll

1. Pse derivati numerik përkeqësohet kur $h$ bëhet tepër i vogël?
2. Si ndryshon hapi optimal kur të dhënat kanë zhurmë?
3. Pse Simpson-i dhe trapezi kanë të njëjtën klasë $O(n)$, por jo të njëjtën kosto për tolerancë?
4. Kur është i besueshëm vlerësimi Richardson?


# Pjesa II — GNU Octave

## 9. Derivati dhe lakorja e gabimit

```octave
x0=1.0; h=logspace(-16,-1,250);
dp=(sin(x0+h)-sin(x0))./h;
dq=(sin(x0+h)-sin(x0-h))./(2*h);
ep=abs(dp-cos(x0)); eq=abs(dq-cos(x0));
loglog(h,ep,h,eq); grid on;
xlabel('h'); ylabel('gabimi'); legend('perpara','qendrore');
[~,i]=min(eq); fprintf('h optimal = %.3e\n',h(i));
```

## 10. Trapezi dhe Simpson-i

```octave
f=@(x) sin(x); a=0; b=pi; n=32;
x=linspace(a,b,n+1); h=(b-a)/n; y=f(x);
T=h*(0.5*y(1)+sum(y(2:end-1))+0.5*y(end));
S=h/3*(y(1)+y(end)+4*sum(y(2:2:end-1))+2*sum(y(3:2:end-2)));
fprintf('T=%.15f, S=%.15f, sakte=2\n',T,S);
```


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = "import numpy as np\n\ndef simpson(f,a,b,n):\n    if n%2: raise ValueError('n çift')\n    x=np.linspace(a,b,n+1); y=f(x); h=(b-a)/n\n    return h/3*(y[0]+y[-1]+4*y[1:-1:2].sum()+2*y[2:-1:2].sum())\nfor n in [4,8,16,32]: print(n,simpson(np.sin,0,np.pi,n))\n"
script_path = script_dir / 'laboratori_05_diferencimi_integrimi.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_05_diferencimi_integrimi.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Nxora formulën qendrore nga Taylor-i.
- [ ] Identifikova minimumin e lakores së gabimit.
- [ ] Nxora kostot ndaj tolerancës.
- [ ] Përdora Richardson-in vetëm në regjimin asimptotik.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
